In [3]:
import sys
sys.path.append('..')
from genoml.utils import *

In [23]:
cell_types = ['HEK293T', 'HeLa-S3', 'HepG2', 'HL-60', 'Jurkat', 'K562', 'MCF-7', 'PANC-1', 'Raji']
cell_names = ['K562_Leukemia_Cell', 'HepG2_Hepatocellular_Carcinoma', 'HEK293T_Epithelium_Embryonic_Kidney', 'HEK293_Epithelium_Embryonic_Kidney', 'Jurkat_T_Lymphocyte_Blood', 'Raji_Lymphoblast', 'GM12878_B_Lymphocyte_Blood', 'PANC-1_Pancreatic_ductal', 'HeLa-S3_Epithelium_Cervix', 'MCF-7_Epithelium_Mammary_Gland', 'HL-60_Leukemia_Cell']

In [24]:
assays = ['DNase', 'H3K4me3', 'H3K27ac', 'CTCF']

In [25]:
MPRA_df = pd.read_csv('../data/TFBU_9cell/data.tsv', sep='\t')
notna_indice = MPRA_df[MPRA_df[cell_types].notna().all(axis=1)].index
MPRA_df = MPRA_df.loc[notna_indice]
data = MPRA_df.drop(columns=['id', 'seq'])
data = (data - data.mean(axis=0)) / data.std(axis=0)
data.describe()
mean = data.mean(axis=1)  # 每条序列跨细胞均值（忽略 NaN）
for cell_type in cell_types:
    diff = (data[cell_type] - mean).abs()
    thr = diff.quantile(0.8)
    mask = (diff >= thr)
    data[f"{cell_type}_specific"] = mask
    

In [26]:
pred_array = load_h5('../predict_CRE_activity/outputs/TFBU_MPRA_Sei_pred.h5')

file ../predict_CRE_activity/outputs/TFBU_MPRA_Sei_pred.h5 has keys: ['data']


In [27]:
df = pd.read_csv('../data/Sei/Sei_tracks_info.csv')
df_pivot = df.pivot_table(
    values="index", 
    index="cell_type", 
    columns="assay", 
    aggfunc=list,
)
df_pivot = df_pivot.map(lambda x: x if isinstance(x, list) else [])
print(df_pivot.shape)
print(df_pivot.loc[cell_names, assays].map(len))

(1338, 1169)
assay                                DNase  H3K4me3  H3K27ac  CTCF
cell_type                                                         
K562_Leukemia_Cell                      58       91       51    29
HepG2_Hepatocellular_Carcinoma           9        8        9    17
HEK293T_Epithelium_Embryonic_Kidney      1       10        1     0
HEK293_Epithelium_Embryonic_Kidney       0       17        5    12
Jurkat_T_Lymphocyte_Blood                3       18        9     2
Raji_Lymphoblast                         0        8        9     0
GM12878_B_Lymphocyte_Blood               8       15       24    17
PANC-1_Pancreatic_ductal                 3        2        6     4
HeLa-S3_Epithelium_Cervix                6        3        1     7
MCF-7_Epithelium_Mammary_Gland          18       56       66    54
HL-60_Leukemia_Cell                      3        6        2     2


In [28]:
VEF_df = pd.DataFrame()

for i, cell_name in enumerate(cell_names):
    for j, assay in enumerate(assays):
        indice = df_pivot.loc[cell_name, assay]
        if len(indice) == 0:
            VEF_df[f'{cell_name}_{assay}'] = np.nan
        else:
            pred = logit(pred_array[:, indice], eps=1e-6).mean(1)
            VEF_df[f'{cell_name}_{assay}'] = pred

VEF_df = VEF_df.loc[notna_indice]
VEF_df

,K562_Leukemia_Cell_DNase,K562_Leukemia_Cell_H3K4me3,K562_Leukemia_Cell_H3K27ac,K562_Leukemia_Cell_CTCF,HepG2_Hepatocellular_Carcinoma_DNase,HepG2_Hepatocellular_Carcinoma_H3K4me3,HepG2_Hepatocellular_Carcinoma_H3K27ac,HepG2_Hepatocellular_Carcinoma_CTCF,HEK293T_Epithelium_Embryonic_Kidney_DNase,HEK293T_Epithelium_Embryonic_Kidney_H3K4me3,HEK293T_Epithelium_Embryonic_Kidney_H3K27ac,HEK293T_Epithelium_Embryonic_Kidney_CTCF,HEK293_Epithelium_Embryonic_Kidney_DNase,HEK293_Epithelium_Embryonic_Kidney_H3K4me3,HEK293_Epithelium_Embryonic_Kidney_H3K27ac,HEK293_Epithelium_Embryonic_Kidney_CTCF,Jurkat_T_Lymphocyte_Blood_DNase,Jurkat_T_Lymphocyte_Blood_H3K4me3,Jurkat_T_Lymphocyte_Blood_H3K27ac,Jurkat_T_Lymphocyte_Blood_CTCF,Raji_Lymphoblast_DNase,Raji_Lymphoblast_H3K4me3,Raji_Lymphoblast_H3K27ac,Raji_Lymphoblast_CTCF,GM12878_B_Lymphocyte_Blood_DNase,GM12878_B_Lymphocyte_Blood_H3K4me3,GM12878_B_Lymphocyte_Blood_H3K27ac,GM12878_B_Lymphocyte_Blood_CTCF,PANC-1_Pancreatic_ductal_DNase,PANC-1_Pancreatic_ductal_H3K4me3,PANC-1_Pancreatic_ductal_H3K27ac,PANC-1_Pancreatic_ductal_CTCF,HeLa-S3_Epithelium_Cervix_DNase,HeLa-S3_Epithelium_Cervix_H3K4me3,HeLa-S3_Epithelium_Cervix_H3K27ac,HeLa-S3_Epithelium_Cervix_CTCF,MCF-7_Epithelium_Mammary_Gland_DNase,MCF-7_Epithelium_Mammary_Gland_H3K4me3,MCF-7_Epithelium_Mammary_Gland_H3K27ac,MCF-7_Epithelium_Mammary_Gland_CTCF,HL-60_Leukemia_Cell_DNase,HL-60_Leukemia_Cell_H3K4me3,HL-60_Leukemia_Cell_H3K27ac,HL-60_Leukemia_Cell_CTCF
0,-1.040,-3.891,-4.817,-5.208,-3.989,-4.319,-7.081,-5.430,-5.666,-4.951,-4.474,NaN,NaN,-4.888,-5.303,-4.930,-2.291,-4.881,-6.290,-4.929,NaN,-5.658,-6.398,NaN,-3.754,-5.667,-6.541,-5.985,-1.857,-2.337,-4.504,-5.586,-2.153,-3.007,-2.988,-5.914,-3.037,-4.490,-5.834,-4.799,-2.073,-4.769,-4.075,-6.965
1,-1.950,-4.402,-5.684,-3.336,-3.302,-3.987,-6.188,-3.348,-5.480,-4.794,-4.851,NaN,NaN,-4.387,-4.931,-4.443,-3.372,-5.217,-7.450,-3.899,NaN,-6.482,-7.262,NaN,-4.877,-6.074,-7.219,-4.495,-3.663,-2.642,-5.469,-2.810,-2.938,-3.032,-3.566,-3.809,-4.431,-5.015,-6.633,-3.287,-3.438,-4.544,-3.641,-6.810
2,-2.027,-4.064,-5.699,-4.101,-3.843,-4.087,-6.732,-4.686,-5.558,-4.194,-3.456,NaN,NaN,-3.897,-4.215,-5.037,-3.504,-5.146,-7.334,-4.842,NaN,-6.706,-7.863,NaN,-5.112,-6.216,-8.009,-5.938,-3.606,-1.829,-4.869,-3.832,-2.816,-2.179,-3.568,-5.027,-3.688,-4.064,-5.632,-3.661,-3.712,-4.589,-3.514,-7.527
3,2.372,-7.379,-9.134,-1.971,0.628,-8.886,-12.622,-1.780,-5.777,-8.645,-10.255,NaN,NaN,-9.874,-8.490,-1.873,1.882,-8.733,-10.588,-1.932,NaN,-9.869,-12.097,NaN,-0.262,-9.119,-12.679,-2.639,0.090,-4.710,-9.399,-1.825,-0.169,-7.505,-9.030,-3.289,-0.766,-8.832,-11.138,-2.147,2.533,-9.042,-8.172,-3.086
4,-2.276,-5.439,-6.303,-3.451,-4.871,-5.751,-8.471,-3.702,-5.764,-5.445,-5.210,NaN,NaN,-5.020,-5.872,-3.959,-3.012,-5.992,-7.686,-3.583,NaN,-6.374,-7.252,NaN,-4.486,-6.278,-7.695,-4.096,-3.390,-3.629,-6.033,-2.992,-2.671,-4.131,-4.344,-3.868,-3.566,-5.544,-7.284,-3.121,-2.437,-5.401,-4.211,-5.554
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17995,-1.190,-3.685,-5.917,-3.700,-1.634,-2.470,-5.632,-4.381,-4.698,-2.426,-1.002,NaN,NaN,-1.919,-2.151,-3.738,-0.686,-2.806,-5.457,-2.725,NaN,-5.069,-5.963,NaN,-0.843,-2.386,-5.333,-3.294,-1.509,1.106,-3.653,-3.048,-1.928,-1.448,-2.695,-4.631,-2.237,-2.307,-4.444,-3.038,-1.010,-2.816,-2.054,-5.005
17996,-2.328,-3.965,-5.991,-4.396,-2.229,-2.589,-5.373,-4.828,-4.782,-2.657,-2.419,NaN,NaN,-2.003,-2.746,-4.390,-3.456,-3.879,-6.384,-4.518,NaN,-4.916,-5.538,NaN,-1.892,-2.100,-3.952,-4.039,-2.496,0.315,-3.522,-3.478,-2.909,-2.058,-3.043,-5.390,-3.017,-2.921,-4.485,-3.804,-2.371,-2.758,-2.403,-6.091
17997,-6.083,-7.998,-8.096,-8.847,-5.746,-5.927,-7.261,-8.087,-7.352,-7.162,-6.639,NaN,NaN,-6.071,-7.157,-8.294,-7.426,-8.706,-10.246,-8.859,NaN,-9.298,-10.504,NaN,-8.604,-10.253,-10.890,-10.132,-6.580,-5.351,-7.663,-7.484,-6.172,-5.854,-6.086,-8.468,-7.198

In [29]:
for j, assay in enumerate(assays):
    pearson_df = pd.DataFrame()
    for cell_name in cell_names:
        for cell_type in cell_types:
            pred = VEF_df[f'{cell_name}_{assay}']
            true = MPRA_df[f'{cell_type}']
            r, _ = pearson(pred, true)
            pearson_df.loc[f'{cell_name}_{assay}', f'{cell_type}'] = r
    print(assay, pearson_df)

DNase                                            HEK293T  HeLa-S3  HepG2  HL-60  Jurkat  K562  MCF-7  PANC-1  Raji
K562_Leukemia_Cell_DNase                     0.554    0.472  0.555  0.442   0.569 0.678  0.404   0.563 0.581
HepG2_Hepatocellular_Carcinoma_DNase         0.361    0.332  0.524  0.266   0.351 0.443  0.287   0.376 0.378
HEK293T_Epithelium_Embryonic_Kidney_DNase    0.252    0.120  0.213  0.107   0.242 0.320  0.175   0.202 0.288
HEK293_Epithelium_Embryonic_Kidney_DNase       NaN      NaN    NaN    NaN     NaN   NaN    NaN     NaN   NaN
Jurkat_T_Lymphocyte_Blood_DNase              0.488    0.410  0.444  0.394   0.576 0.583  0.396   0.468 0.578
Raji_Lymphoblast_DNase                         NaN      NaN    NaN    NaN     NaN   NaN    NaN     NaN   NaN
GM12878_B_Lymphocyte_Blood_DNase             0.519    0.450  0.502  0.419   0.539 0.598  0.435   0.518 0.580
PANC-1_Pancreatic_ductal_DNase               0.506    0.463  0.543  0.419   0.477 0.579  0.408   0.521 0.504
HeLa-S3_Epith

In [30]:
for j, assay in enumerate(assays):
    pearson_df = pd.DataFrame()
    for i, cell_type in enumerate(cell_types):
        for ii, cell_name in enumerate(cell_names):
            pred = VEF_df[f'{cell_name}_{assay}']
            true = data[f'{cell_type}']

            indice = data[f'{cell_type}_specific']
            pred = pred.loc[indice]
            true = true.loc[indice]
            r, _ = pearson(pred, true)
            pearson_df.loc[f'{cell_name}_{assay}', f'{cell_type}'] = r
    print(pearson_df)

                                           HEK293T  HeLa-S3  HepG2  HL-60  Jurkat  K562  MCF-7  PANC-1  Raji
K562_Leukemia_Cell_DNase                     0.355    0.311  0.321  0.324   0.429 0.666  0.203   0.389 0.417
HepG2_Hepatocellular_Carcinoma_DNase         0.130    0.186  0.427  0.120   0.154 0.376  0.159   0.192 0.152
HEK293T_Epithelium_Embryonic_Kidney_DNase    0.233   -0.001  0.158 -0.076   0.157 0.362  0.157   0.049 0.296
HEK293_Epithelium_Embryonic_Kidney_DNase       NaN      NaN    NaN    NaN     NaN   NaN    NaN     NaN   NaN
Jurkat_T_Lymphocyte_Blood_DNase              0.292    0.255  0.153  0.256   0.498 0.538  0.240   0.204 0.514
Raji_Lymphoblast_DNase                         NaN      NaN    NaN    NaN     NaN   NaN    NaN     NaN   NaN
GM12878_B_Lymphocyte_Blood_DNase             0.320    0.301  0.274  0.260   0.388 0.530  0.285   0.316 0.474
PANC-1_Pancreatic_ductal_DNase               0.315    0.353  0.387  0.312   0.283 0.527  0.241   0.394 0.289
HeLa-S3_Epithelium_

In [ ]:
cell_types = ['HepG2', 'HEK293T', 'Jurkat', 'PANC-1', 'HeLa-S3']
assays = ['DNase', 'H3K4me3', 'H3K27ac', 'CTCF']

cell_type_name_map = {
    'HepG2': 'HepG2_Hepatocellular_Carcinoma',
    'HEK293T': 'HEK293T_Epithelium_Embryonic_Kidney',
    'Jurkat': 'Jurkat_T_Lymphocyte_Blood',
    # 'Raji': 'Raji_Lymphoblast',
    'PANC-1': 'PANC-1_Pancreatic_ductal',
    'HeLa-S3': 'HeLa-S3_Epithelium_Cervix',
}


df_track = pd.DataFrame()

for cell_type in cell_types:
    cell_name = cell_type_name_map[cell_type]
    df_track.loc[cell_type, assays] = df_pivot.loc[cell_name, assays]

df_track.loc['HEK293T', 'CTCF'] = df_pivot.loc['HEK293T_Epithelium_Embryonic_Kidney', 'CTCF'] + df_pivot.loc['HEK293_Epithelium_Embryonic_Kidney', 'CTCF']

print(df_pivot.loc[cell_names, assays].map(len))
print(df_track.loc[cell_types, assays].map(len))

assay                                DNase  H3K4me3  H3K27ac  CTCF
cell_type                                                         
K562_Leukemia_Cell                      58       91       51    29
HepG2_Hepatocellular_Carcinoma           9        8        9    17
HEK293T_Epithelium_Embryonic_Kidney      1       10        1     0
HEK293_Epithelium_Embryonic_Kidney       0       17        5    12
Jurkat_T_Lymphocyte_Blood                3       18        9     2
Raji_Lymphoblast                         0        8        9     0
GM12878_B_Lymphocyte_Blood               8       15       24    17
PANC-1_Pancreatic_ductal                 3        2        6     4
HeLa-S3_Epithelium_Cervix                6        3        1     7
         DNase  H3K4me3  H3K27ac  CTCF
HepG2        9        8        9    17
HEK293T      1       10        1    12
Jurkat       3       18        9     2
PANC-1       3        2        6     4
HeLa-S3      6        3        1     7


In [28]:
VEF_df = pd.DataFrame()

for i, cell_type in enumerate(cell_types):
    for j, assay in enumerate(assays):
        indice = df_track.loc[cell_type, assay]
        if len(indice) == 0:
            VEF_df[f'{cell_type}_{assay}'] = np.nan
        else:
            pred = logit(pred_array[:, indice], eps=1e-6).mean(1)
            VEF_df[f'{cell_type}_{assay}'] = pred
VEF_df

,HepG2_DNase,HepG2_H3K4me3,HepG2_H3K27ac,HepG2_CTCF,HEK293T_DNase,HEK293T_H3K4me3,HEK293T_H3K27ac,HEK293T_CTCF,Jurkat_DNase,Jurkat_H3K4me3,Jurkat_H3K27ac,Jurkat_CTCF,PANC-1_DNase,PANC-1_H3K4me3,PANC-1_H3K27ac,PANC-1_CTCF,HeLa-S3_DNase,HeLa-S3_H3K4me3,HeLa-S3_H3K27ac,HeLa-S3_CTCF
0,-3.989,-4.319,-7.081,-5.430,-5.666,-4.951,-4.474,-4.930,-2.291,-4.881,-6.290,-4.929,-1.857,-2.337,-4.504,-5.586,-2.153,-3.007,-2.988,-5.914
1,-3.302,-3.987,-6.188,-3.348,-5.480,-4.794,-4.851,-4.443,-3.372,-5.217,-7.450,-3.899,-3.663,-2.642,-5.469,-2.810,-2.938,-3.032,-3.566,-3.809
2,-3.843,-4.087,-6.732,-4.686,-5.558,-4.194,-3.456,-5.037,-3.504,-5.146,-7.334,-4.842,-3.606,-1.829,-4.869,-3.832,-2.816,-2.179,-3.568,-5.027
3,0.628,-8.886,-12.622,-1.780,-5.777,-8.645,-10.255,-1.873,1.882,-8.733,-10.588,-1.932,0.090,-4.710,-9.399,-1.825,-0.169,-7.505,-9.030,-3.289
4,-4.871,-5.751,-8.471,-3.702,-5.764,-5.445,-5.210,-3.959,-3.012,-5.992,-7.686,-3.583,-3.390,-3.629,-6.033,-2.992,-2.671,-4.131,-4.344,-3.868
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17995,-1.634,-2.470,-5.632,-4.381,-4.698,-2.426,-1.002,-3.738,-0.686,-2.806,-5.457,-2.725,-1.509,1.106,-3.653,-3.048,-1.928,-1.448,-2.695,-4.631
17996,-2.229,-2.589,-5.373,-4.828,-4.782,-2.657,-2.419,-4.390,-3.456,-3.879,-6.384,-4.518,-2.496,0.315,-3.522,-3.478,-2.909,-2.058,-3.043,-5.390
17997,-5.746,-5.927,-7.261,-8.087,-7.352,-7.162,-6.639,-8.294,-7.426,-8.706,-10.246,-8.859,-6.580,-5.351,-7.663,-7.484,-6.172,-5.854,-6.086,-8.468
17998,-5.750,-7.314,-9.137,-6.391,-6.519,-6.570,-6.450,-6.029,-4.447,-8.009,-9.207,-6.521,-4.320,-5.891,-7.369,-6.423,-3.216,-5.159,-4.729,-6.361


In [ ]:
VEF_df.to_csv('../data/TFBU_MPRA/TFBU_MPRA_Sei_VEF_raw.tsv', sep='\t', index=False)

In [29]:
for j, assay in enumerate(assays):
    pearson_df = pd.DataFrame()
    for ct1 in cell_types:
        for ct2 in cell_types:
            true = MPRA_df[f'{ct1}']
            pred = VEF_df[f'{ct2}_{assay}']
            r, _ = pearson(pred, true)
            pearson_df.loc[f'{ct2}_{assay}', f'{ct1}'] = r
    print(assay)
    print(pearson_df)

DNase
               HepG2  HEK293T  Jurkat  PANC-1  HeLa-S3
HepG2_DNase    0.522    0.364   0.336   0.380    0.337
HEK293T_DNase  0.214    0.256   0.214   0.208    0.127
Jurkat_DNase   0.442    0.489   0.537   0.470    0.413
PANC-1_DNase   0.541    0.506   0.461   0.521    0.465
HeLa-S3_DNase  0.512    0.455   0.403   0.478    0.447
H3K4me3
                 HepG2  HEK293T  Jurkat  PANC-1  HeLa-S3
HepG2_H3K4me3    0.434    0.323   0.330   0.301    0.274
HEK293T_H3K4me3  0.339    0.364   0.340   0.357    0.259
Jurkat_H3K4me3   0.386    0.395   0.447   0.383    0.314
PANC-1_H3K4me3   0.399    0.355   0.354   0.373    0.319
HeLa-S3_H3K4me3  0.414    0.364   0.372   0.366    0.350
H3K27ac
                 HepG2  HEK293T  Jurkat  PANC-1  HeLa-S3
HepG2_H3K27ac    0.432    0.251   0.212   0.213    0.185
HEK293T_H3K27ac  0.357    0.328   0.244   0.277    0.201
Jurkat_H3K27ac   0.364    0.353   0.392   0.316    0.264
PANC-1_H3K27ac   0.469    0.422   0.378   0.419    0.340
HeLa-S3_H3K27ac  0.46

In [30]:
VEF_df = (VEF_df - VEF_df.mean()) / VEF_df.std()

In [31]:
VEF_df.to_csv('../data/TFBU_MPRA/TFBU_MPRA_Sei_VEF_zscore.tsv', sep='\t', index=False)